# 01 · Data Prep — Limpieza y exportación
### Vinson Digital · Case Study

Este notebook hace **solo** la carga y limpieza, y **exporta los datos limpios**
a disco para que el notebook de modelado (`02_modeling.ipynb`) los consuma sin
volver a tocar el CSV crudo.

**Salidas que genera:**
- `clean_data_long.parquet` / `.csv` → formato *tidy* (una fila por fecha-producto-calibre).
- `clean_data_wide.parquet` / `.csv` → formato *ancho* (una columna por serie, indexado por fecha), listo para modelar.

**Variable objetivo:** precio *real* semanal.
Brazil HON = promedio de las dos lecturas reales (P Real X / P Real XII);
USA Fillet = columna P Real.

---

## 1. Configuración

In [ ]:
#import sys -- usando entorno para que sea replicable
#print(sys.executable)

/Users/marioromero/case_study/.venv/bin/python


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

CSV = "../data/raw/Dataset_case_study.csv" 
OUT_DIR = '.'                    

## 2. Funciones de carga y limpieza

El CSV trae un encabezado de **3 niveles** (producto / calibre / serie). Lo
aplanamos a `Producto|Calibre|Serie`, derivamos la fecha desde año + semana ISO,
y construimos el formato tidy quedándonos con el **precio real**.

In [2]:
PROD_MAP = {'Brazil HON — Precio neto USD/kg FCA': 'Brazil HON',
            'USA Fillet — Precio neto USD/lb FOB Miami (spot)': 'USA Fillet'}

def load_raw(path):
    """Lee el CSV y aplana el encabezado de 3 niveles a 'Producto|Calibre|Serie'."""
    raw = pd.read_csv(path, header=[0, 1, 2])
    lvl0 = pd.Series([c[0] if 'Unnamed' not in c[0] else None for c in raw.columns]).ffill()
    lvl1 = pd.Series([c[1] if 'Unnamed' not in c[1] else None for c in raw.columns]).ffill()
    lvl2 = [c[2] for c in raw.columns]
    cols = []
    for p, cal, col in zip(lvl0, lvl1, lvl2):
        cols.append(col if col in ('Año','Semana') else f'{PROD_MAP.get(p,p)}|{cal}|{col}')
    raw.columns = cols
    raw['Año'] = raw['Año'].astype(int); raw['Semana'] = raw['Semana'].astype(int)
    return raw

def iso_week_to_date(y, w):
    """Año + semana ISO -> fecha del lunes de esa semana."""
    return pd.Timestamp.fromisocalendar(y, min(w, 52), 1)

def build_tidy(path):
    """Devuelve un dataframe largo: fecha, producto, calibre, unidad, precio (real)."""
    raw = load_raw(path)
    raw['fecha'] = [iso_week_to_date(y, w) for y, w in zip(raw['Año'], raw['Semana'])]
    rec = []
    # Brazil HON: promedio de las dos lecturas reales
    for cal in ['10-12 lb','12-14 lb','14-16 lb']:
        px  = pd.to_numeric(raw[f'Brazil HON|{cal}|P Real X'],   errors='coerce')
        pxi = pd.to_numeric(raw[f'Brazil HON|{cal}|P Real XII'], errors='coerce')
        precio = pd.concat([px, pxi], axis=1).mean(axis=1)
        for f, p in zip(raw['fecha'], precio):
            rec.append(('Brazil HON', cal, 'USD/kg', f, p))
    # USA Fillet: columna P Real
    for cal in ['2/3','3/4','4/5']:
        precio = pd.to_numeric(raw[f'USA Fillet|{cal}|P Real'], errors='coerce')
        for f, p in zip(raw['fecha'], precio):
            rec.append(('USA Fillet', cal, 'USD/lb', f, p))
    return (pd.DataFrame(rec, columns=['producto','calibre','unidad','fecha','precio'])
              .sort_values(['producto','calibre','fecha']).reset_index(drop=True))

## 3. Construcción del dataset tidy

In [3]:
tidy = build_tidy(CSV)
print('Filas:', len(tidy), '| Series:', tidy.groupby(['producto','calibre']).ngroups)
tidy.head()

Filas: 1326 | Series: 6


,producto,calibre,unidad,fecha,precio
0,Brazil HON,10-12 lb,USD/kg,2022-01-03,7.6000
1,Brazil HON,10-12 lb,USD/kg,2022-01-10,7.6058
2,Brazil HON,10-12 lb,USD/kg,2022-01-17,7.6062
3,Brazil HON,10-12 lb,USD/kg,2022-01-24,7.6333
4,Brazil HON,10-12 lb,USD/kg,2022-01-31,8.1000


### Cobertura y completitud (control de calidad)

In [4]:
tidy.groupby(['producto','calibre']).agg(
    n=('precio','size'), no_nulos=('precio', lambda x: x.notna().sum()),
    desde=('fecha','min'), hasta=('fecha','max'),
    min=('precio','min'), max=('precio','max'), media=('precio','mean')).round(2)

n  no_nulos      desde      hasta   min    max  media
producto   calibre                                                          
Brazil HON 10-12 lb  221       216 2022-01-03 2026-03-23  5.35  10.04   7.46
           12-14 lb  221       217 2022-01-03 2026-03-23  5.45  10.07   7.56
           14-16 lb  221       216 2022-01-03 2026-03-23  5.56  10.05   7.61
USA Fillet 2/3       221       216 2022-01-03 2026-03-23  5.08   7.57   6.14
           3/4       221       217 2022-01-03 2026-03-23  5.13   7.60   6.22
           4/5       221       212 2022-01-03 2026-03-23  5.20   7.78   6.34

## 4. Imputación de huecos cortos y formato ancho

Reindexamos a frecuencia semanal (`W-MON`), interpolamos linealmente los huecos
cortos (≤3 semanas) y armamos la tabla **ancha**: una columna por serie,
indexada por fecha. Este es el formato más cómodo para alimentar modelos.

In [5]:
SERIES = [('Brazil HON','10-12 lb'),('Brazil HON','12-14 lb'),('Brazil HON','14-16 lb'),
          ('USA Fillet','2/3'),('USA Fillet','3/4'),('USA Fillet','4/5')]

def get_series(producto, calibre):
    s = tidy[(tidy.producto==producto)&(tidy.calibre==calibre)] \
        .set_index('fecha')['precio'].asfreq('W-MON')
    return s.interpolate('linear', limit=3, limit_area='inside')

wide = pd.DataFrame({f'{p}|{c}': get_series(p, c) for p, c in SERIES})
wide.index.name = 'fecha'
print('Formato ancho:', wide.shape, '| rango', wide.index.min().date(), '→', wide.index.max().date())
print('Nulos restantes por columna:'); print(wide.isna().sum())
wide.tail()

Formato ancho: (221, 6) | rango 2022-01-03 → 2026-03-23
Nulos restantes por columna:
Brazil HON|10-12 lb    4
Brazil HON|12-14 lb    4
Brazil HON|14-16 lb    4
USA Fillet|2/3         5
USA Fillet|3/4         4
USA Fillet|4/5         7
dtype: int64


,Brazil HON|10-12 lb,Brazil HON|12-14 lb,Brazil HON|14-16 lb,USA Fillet|2/3,USA Fillet|3/4,USA Fillet|4/5
fecha,,,,,,
2026-02-23,6.9773,7.06895,7.17475,6.4627,6.5609,6.6387
2026-03-02,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-09,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-16,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-23,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Exportación de datos limpios

Guardamos en **parquet** (recomendado: conserva tipos y fechas) y en **csv**
(portable). El notebook de modelado leerá estos archivos.

In [6]:
# Tidy (largo) — interpolado para consistencia con el formato ancho
tidy_clean = (wide.reset_index()
                  .melt(id_vars='fecha', var_name='serie', value_name='precio'))
tidy_clean[['producto','calibre']] = tidy_clean['serie'].str.split('|', expand=True)
tidy_clean['unidad'] = np.where(tidy_clean['producto']=='Brazil HON','USD/kg','USD/lb')
tidy_clean = tidy_clean[['fecha','producto','calibre','unidad','serie','precio']]

# Guardar ambos formatos
tidy_clean.to_parquet(f'{OUT_DIR}/clean_data_long.parquet', index=False)
#tidy_clean.to_csv(f'{OUT_DIR}/clean_data_long.csv', index=False)
wide.to_parquet(f'{OUT_DIR}/clean_data_wide.parquet')
# wide.to_csv(f'{OUT_DIR}/clean_data_wide.csv') # no usare csv esta vez pero lo dejo comentado por si alguien lo prefiere

print('Archivos guardados:')

Archivos guardados:
